### DATA Prepping

In [ ]:
!pip install --quiet presidio-analyzer
!pip --quiet install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import json
import random
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
from presidio_analyzer import AnalyzerEngine

In [ ]:
from utils import *

In [ ]:
df_full = pd.read_csv('PII43k.csv', on_bad_lines='skip')

df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME_1] to send ...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [FULLNAME_1] who wants...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [FULLNAME...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [F...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."


In [ ]:
cleaned_matches, unique_matches = get_template_tokens(df_full)

def replace_unique_tokens(text, tokens):
	for token in tokens:
		# Match either an underscore with one or more digits or with 'N'
		pattern = r'\[' + token + r'_(?:\d+|N)\]'
		# Replace with the token in square brackets (e.g., "[NAME]")
		text = re.sub(pattern, f'[{token}]', text)
	return text

df_full['Template'] = df_full['Template'].apply(lambda t: replace_unique_tokens(t, cleaned_matches))

# view the first 5 rows of the 'Template' column
df_full['Template'].head()

0    In our video conference, discuss the role of e...
1    Could you draft a letter for [NAME] to send to...
2    Discuss the options for [FULLNAME] who wants t...
3    13. Write a press release announcing [FULLNAME...
4    9. Develop an inventory management plan for [F...
Name: Template, dtype: object

In [ ]:
df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [FULLNAME] who wants t...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [FULLNAME...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [F...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."


In [ ]:
cleaned_matches, unique_matches = get_template_tokens(df_full)
print(unique_matches)
print(len(unique_matches))

{'BIC', 'MASKEDNUMBER', 'IPV4', 'EMAIL', 'STATE', 'FULLNAME', 'ETHEREUMADDRESS', 'CITY', 'CURRENCY', 'URL', 'DISPLAYNAME', 'ACCOUNTNUMBER', 'BITCOINADDRESS', 'ORDINALDIRECTION', 'JOBTITLE', 'SECONDARYADDRESS', 'CURRENCYNAME', 'PASSWORD', 'IP', 'COUNTY', 'JOBAREA', 'CURRENCYSYMBOL', 'LITECOINADDRESS', 'CREDITCARDISSUER', 'CREDITCARDNUMBER', 'NUMBER', 'MAC', 'JOBTYPE', 'IBAN', 'USERNAME', 'NEARBYGPSCOORDINATE', 'SEXTYPE', 'FIRSTNAME', 'AMOUNT', 'IPV6', 'NAME', 'CREDITCARDCVV', 'ZIPCODE', 'GENDER', 'CURRENCYCODE', 'STREET', 'LASTNAME', 'USERAGENT', 'PIN', 'JOBDESCRIPTOR', 'STREETADDRESS', 'BUILDINGNUMBER', 'ACCOUNTNAME', 'SEX'}
49


In [ ]:
# Display a few rows to verify the changes
print(df_full[['Template', 'Filled Template']].head())

                                            Template  \
0  In our video conference, discuss the role of e...   
1  Could you draft a letter for [NAME] to send to...   
2  Discuss the options for [FULLNAME] who wants t...   
3  13. Write a press release announcing [FULLNAME...   
4  9. Develop an inventory management plan for [F...   

                                     Filled Template  
0  In our video conference, discuss the role of e...  
1  Could you draft a letter for Dietrich, Schulis...  
2  Discuss the options for Jeffery Pfeffer who wa...  
3  13. Write a press release announcing Gayle Wat...  
4  9. Develop an inventory management plan for Ev...  


In [ ]:
# Replace any occurrence of "[FULLNAME]" (with optional suffix) in the Template 
# with "[NAME] [NAME]"

df_full['Template'] = df_full['Template'].apply(
	lambda t: re.sub(r'\[FULLNAME(?:_(?:\d+|N))?\]', "[NAME] [NAME]", t)
)

# Verify the changes
print(df_full['Template'].head())

0    In our video conference, discuss the role of e...
1    Could you draft a letter for [NAME] to send to...
2    Discuss the options for [NAME] [NAME] who want...
3    13. Write a press release announcing [NAME] [N...
4    9. Develop an inventory management plan for [N...
Name: Template, dtype: object


In [ ]:
# Display a few examples with the token "[FIRSTNAME]" in the Template
fname_examples = df_full[df_full['Template'].str.contains(r'\[FIRSTNAME_1\]', na=False)]

if fname_examples.empty:
	print("No rows found with the token [FIRSTNAME_1]")
else:
	for i, row in fname_examples.head(5).iterrows():
		print("Template:", row["Template"])
		print("Filled Template:", row["Filled Template"])
		print("-" * 60)

No rows found with the token [FIRSTNAME_1]


In [ ]:
token_mapping = {
  "LOCATION": [ 
    "STREETADDRESS",
    "SECONDARYADDRESS",
    "BUILDINGNUMBER",
    "STREET",
    "CITY",
    "STATE",
    "COUNTY",
    "NEARBYGPSCOORDINATE",
    "ZIPCODE"
  ],
  "GENDER": [
    "SEXTYPE",
    "SEX"
  ],
  "USERNAME":[
    "DISPLAYNAME"
  ],
  "NAME": [
    "NAME",
    "FIRSTNAME",
    "LASTNAME"
  ],
  "IP": [
    "IPV4",
    "IP",
    "IPV6",
    "MAC"
  ],
  "JOB": [
    "JOBDESCRIPTOR",
    "JOBTYPE",
    "JOBTITLE",
    "JOBAREA"
  ],
  "MASKEDNUMBER": [
    "NUMBER",
    "AMOUNT",
    "ACCOUNTNUMBER",
    "CREDITCARDCVV",
    "PIN",
    "BUILDINGNUMBER"
  ]
}

In [ ]:
def replace_tokens_with_category(text, mapping):
	for category, tokens in mapping.items():
		for token in tokens:
			# Replace tokens with a suffix (e.g., [CITY_1] or [CITY_N])
			pattern = r'\[' + token + r'_(?:\d+|N)\]'
			text = re.sub(pattern, f'[{category.upper()}]', text)
			# Replace tokens without a suffix (e.g., [CITY])
			pattern_no_suffix = r'\[' + token + r'\]'
			text = re.sub(pattern_no_suffix, f'[{category.upper()}]', text)
	return text

# Update the 'Template' column in df_full
df_full['Template'] = df_full['Template'].apply(lambda t: replace_tokens_with_category(t, token_mapping))
print(df_full['Template'].head())

_, unique_matches = get_template_tokens(df_full)
print(unique_matches)
print(len(unique_matches))

0    In our video conference, discuss the role of e...
1    Could you draft a letter for [NAME] to send to...
2    Discuss the options for [NAME] [NAME] who want...
3    13. Write a press release announcing [NAME] [N...
4    9. Develop an inventory management plan for [N...
Name: Template, dtype: object
{'BIC', 'MASKEDNUMBER', 'EMAIL', 'LOCATION', 'ETHEREUMADDRESS', 'CURRENCY', 'URL', 'ORDINALDIRECTION', 'BITCOINADDRESS', 'CURRENCYNAME', 'PASSWORD', 'IP', 'CURRENCYSYMBOL', 'LITECOINADDRESS', 'CREDITCARDISSUER', 'CREDITCARDNUMBER', 'IBAN', 'USERNAME', 'NAME', 'GENDER', 'CURRENCYCODE', 'USERAGENT', 'ACCOUNTNAME', 'JOB'}
24


In [ ]:
df_full

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [NAME] [NAME] who want...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [NAME] [N...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [N...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."
...,...,...,...,...
42754,Write a blog post for [NAME] about the role of...,Write a blog post for Stanton LLC about the ro...,"['write', 'a', 'blog', 'post', 'for', 'stanton...","['O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', ..."
42755,14. Calculate the return on investment for [NA...,14. Calculate the return on investment for Con...,"['14', '.', 'calculate', 'the', 'return', 'on'...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-NA..."
42756,Please write an email to [NAME] [NAME] at [EMA...,Please write an email to Roberta Gutmann V at ...,"['please', 'write', 'an', 'email', 'to', 'robe...","['O', 'O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FUL..."
42757,Can you help me write a project closure report...,Can you help me write a project closure report...,"['can', 'you', 'help', 'me', 'write', 'a', 'pr...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [ ]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(df_full["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
18,NAME,55856
3,LOCATION,12118
2,EMAIL,7914
23,JOB,3369
1,MASKEDNUMBER,1477
6,URL,1028
17,USERNAME,713
11,IP,557
5,CURRENCY,350
10,PASSWORD,289


In [ ]:
import re

# List of tokens to replace
remove_tokens = [
    "ORDINALDIRECTION", "ACCOUNTNAME", "CURRENCYSYMBOL",
    "CURRENCYNAME", "CURRENCY", "CURRENCYCODE",
    "CREDITCARDISSUER", "ETHEREUMADDRESS", "LITECOINADDRESS", "BITCOINADDRESS","BIC"
]

def unmask_text(text_with_mask, text_without_mask, tokens):
    general_token_pattern = r'\[([A-Z0-9_]+)(?:_(?:\d+|N))?\]'
    matches = list(re.finditer(general_token_pattern, text_with_mask))
    pattern_parts = []
    pos = 0

    for i, match in enumerate(matches):
        # Append literal text (escaped)
        literal_text = re.escape(text_with_mask[pos:match.start()])
        pattern_parts.append(literal_text)

        token_name = match.group(1)
        lookahead = ""
        if token_name in tokens:
            if i + 1 < len(matches):
                next_literal = text_with_mask[match.end():matches[i+1].start()]
                next_token = matches[i+1].group(1)
                if next_literal == "" and next_token not in tokens:
                    lookahead = "(?=\d)"
            # Use a unique group name for each occurrence
            group_name = f"{token_name}_{i}"
            pattern_parts.append(f"(?P<{group_name}>.+?){lookahead}")
        else:
            pattern_parts.append(".+?")
        pos = match.end()

    pattern_parts.append(re.escape(text_with_mask[pos:]))
    final_pattern = ''.join(pattern_parts)
    match_obj = re.fullmatch(final_pattern, text_without_mask)
    if match_obj:
        return match_obj.groupdict()
    return {}

def consolidate_extracted_values(extracted):
    """
    Convert keys like CREDITCARDISSUER_3, CREDITCARDISSUER_7 into a dictionary
    with key 'CREDITCARDISSUER' mapping to a list of values.
    """
    consolidated = {}
    for key, value in extracted.items():
        base = key.split('_')[0]
        consolidated.setdefault(base, []).append(value)
    return consolidated

def replace_tokens(text_with_mask, extracted_values):
    """
    Replace tokens in text_with_mask with corresponding extracted values.
    For tokens appearing multiple times, use a counter to replace them in order.
    """
    counters = {}
    def replacement(match):
        token_name = match.group(1)
        values = extracted_values.get(token_name, [])
        if not values:
            return ""
        idx = counters.get(token_name, 0)
        counters[token_name] = idx + 1
        return values[idx]
    token_regex = r'\[(' + '|'.join(remove_tokens) + r')(?:_(?:\d+|N))?\]'
    return re.sub(token_regex, replacement, text_with_mask)

# Example usage within your loop:
for i, row in df_full.iterrows():
    masked = row['Template']
    filled = row['Filled Template']
    extracted = unmask_text(masked, filled, remove_tokens)
    
    # Consolidate extracted values so that tokens map to lists
    if extracted:
        consolidated = consolidate_extracted_values(extracted)
        unmasked_text = replace_tokens(masked, consolidated)
    else:
        unmasked_text = masked

    if i == -1:
        print("Extracted values (consolidated):", consolidated)
        print("row:", i)
        print("Masked text:", masked)
        print("Filled text:", filled)
        print("-" * 50)
        print("Unmasked text:", unmasked_text)
        print("-" * 50)
        break

    if extracted != {}:
        print("Extracted values (consolidated):", consolidated)
        print("Masked text:", masked)
        print("Filled text:", filled)
        print("-" * 50)
        print("Unmasked text:", unmasked_text)
        print("-" * 50)
        df_full.at[i, 'Template'] = unmasked_text


<>:28: SyntaxWarning: invalid escape sequence '\d'
<>:28: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_52031/3775163306.py:28: SyntaxWarning: invalid escape sequence '\d'
  lookahead = "(?=\d)"


Extracted values (consolidated): {'BIC': ['FXLOCRGC']}
Masked text: I'm curious about the banking laws related to [IBAN] and [BIC] codes for international transactions. Can you help?
Filled text: I'm curious about the banking laws related to PK82QFFA7074540111686602 and FXLOCRGC codes for international transactions. Can you help?
--------------------------------------------------
Unmasked text: I'm curious about the banking laws related to [IBAN] and FXLOCRGC codes for international transactions. Can you help?
--------------------------------------------------
Extracted values (consolidated): {'BIC': ['KKHYCZO5']}
Masked text: Research the legal aspects of international wire transfers and create a report for [NAME] that includes [IBAN] and [BIC].
Filled text: Research the legal aspects of international wire transfers and create a report for Prosacco and Sons that includes IL513608700509102101028 and KKHYCZO5.
--------------------------------------------------
Unmasked text: Research th

In [ ]:
df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [NAME] [NAME] who want...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [NAME] [N...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [N...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."


In [ ]:
id = 2375
masked = df_full["Template"][id]
filled = df_full["Filled Template"][id]
extracted_values = unmask_text(masked, filled, remove_tokens)
unmasked_text = replace_tokens(masked, extracted_values)

print("Template:")
print(masked)
print("\nFilled Template:")
print(filled)

Template:
Can you help me prepare a project budget for [NAME] with an estimated cost of Nuevo Sol[MASKEDNUMBER]?

Filled Template:
Can you help me prepare a project budget for Kreiger - Daugherty with an estimated cost of Nuevo Sol730.32?


In [ ]:
id = 39654
masked = df_full["Template"][id]
filled = df_full["Filled Template"][id]
extracted_values = unmask_text(masked, filled, remove_tokens)
unmasked_text = replace_tokens(masked, extracted_values)

print("Template:")
print(masked)
print("\nFilled Template:")
print(filled)

Template:
Please draft a memo to [NAME] on the recent court ruling regarding diners_club and its impact on banking law.

Filled Template:
Please draft a memo to Zemlak and Sons on the recent court ruling regarding diners_club and its impact on banking law.


In [ ]:
id = 38860
masked = df_full["Template"][id]
filled = df_full["Filled Template"][id]
extracted_values = unmask_text(masked, filled, remove_tokens)
unmasked_text = replace_tokens(masked, extracted_values)

print("Template:")
print(masked)
print("\nFilled Template:")
print(filled)

Template:
Can you help me understand the legal aspects of managing multiple bank accounts with different maestro and switch?

Filled Template:
Can you help me understand the legal aspects of managing multiple bank accounts with different maestro and switch?


In [ ]:
# Count the labeled items in the DataFrame
total_labeled_items = count_labeled_items(df_full)
print(f"Total number of labeled items: {total_labeled_items}")
print("Number of rows in df_full:", df_full.shape[0])

Total number of labeled items: 83957
Number of rows in df_full: 42759


In [ ]:
# Find rows in annotations_df where the Type is BIC
bic_rows = annotations_df[annotations_df['Type'] == 'BIC']
if not bic_rows.empty:
	print("Rows with BIC type:")
	print(bic_rows)
	
	# Get the corresponding line in aws_prepping
	if 'Line' in bic_rows.columns:
		line_numbers = bic_rows['Line'].unique()
		print("\nCorresponding lines in aws_prepping:")
		for line_num in line_numbers:
			if line_num < len(aws_prepping):
				print(f"\nLine {line_num}:")
				print(f"Template: {aws_prepping['Template'].iloc[line_num]}")
				print(f"Filled Template: {aws_prepping['Filled Template'].iloc[line_num]}")
				print(f"Masked Filled Template: {aws_prepping['Masked Filled Template'].iloc[line_num]}")
else:
	print("No rows with BIC type found in annotations_df.")

Rows with BIC type:
                   File   Line  Begin Offset  End Offset Type
71629  aws_ann_data.csv  36484             0         105  BIC

Corresponding lines in aws_prepping:

Line 36484:
Template: I need help understanding the consumer protection laws that apply to transactions made using BIC [BIC].
Filled Template: I need help understanding the consumer protection laws that apply to transactions made usingEQYEHMCMIC_1].
Masked Filled Template: [BIC].


In [ ]:
df_full.to_csv("ourdata.csv", index=False)

## AWS Comprehend dataprepping START


In [ ]:
# AWS Comprehend dataprepping
aws_prepping = df_full.copy()
aws_prepping = aws_prepping.drop(columns=['Tokenised Filled Template', 'Tokens'] )
aws_prepping

,Template,Filled Template
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis..."
2,Discuss the options for [NAME] [NAME] who want...,Discuss the options for Jeffery Pfeffer who wa...
3,13. Write a press release announcing [NAME] [N...,13. Write a press release announcing Gayle Wat...
4,9. Develop an inventory management plan for [N...,9. Develop an inventory management plan for Ev...
...,...,...
42754,Write a blog post for [NAME] about the role of...,Write a blog post for Stanton LLC about the ro...
42755,14. Calculate the return on investment for [NA...,14. Calculate the return on investment for Con...
42756,Please write an email to [NAME] [NAME] at [EMA...,Please write an email to Roberta Gutmann V at ...
42757,Can you help me write a project closure report...,Can you help me write a project closure report...


In [ ]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(aws_prepping["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
18,NAME,55856
3,LOCATION,12118
2,EMAIL,7914
23,JOB,3369
1,MASKEDNUMBER,1477
6,URL,1028
17,USERNAME,713
11,IP,557
10,PASSWORD,289
19,GENDER,255


In [ ]:
import re
import pandas as pd

def transform_dataframe(df, doc_name="aws_prepping.csv"):
    # This regex matches placeholders like [NAME], [EMAIL], [ANYTHING_123], etc.
    placeholder_pattern = re.compile(r'(\[[^\]]+\])')
    annotation_rows = []

    for i, row in df.iterrows():
        template_text = str(row["Template"])
        filled_text   = str(row["Filled Template"])
        
        # Split template into chunks + placeholders, e.g.:
        # ["Could you draft a letter for ", "[NAME]", " to send to ", "[EMAIL]", ...]
        parts = placeholder_pattern.split(template_text)
        
        # We'll keep track of where we are in the filled_text
        filled_pos = 0
        
        if i == 25:
            print("Row 25 details:")
            print(row.to_dict())
            print("Parts:", parts)

        # Walk through each chunk or placeholder
        for idx, part in enumerate(parts):
            # Check if 'part' is a placeholder (e.g., "[NAME]")
            if placeholder_pattern.match(part):
                # Extract the entity type from the placeholder text, e.g. "[NAME]" -> "NAME"
                entity_type = part.strip("[]")
                
                # Find the next literal chunk in parts (the text that comes after this placeholder)
                next_chunk = ""
                for j in range(idx + 1, len(parts)):
                    if not placeholder_pattern.match(parts[j]):
                        next_chunk = parts[j]
                        break
                
                # If there's no next chunk, the replaced text goes until the end of filled_text
                if next_chunk == "":
                    replaced_text = filled_text[filled_pos:]
                    begin_offset = filled_pos
                    end_offset   = filled_pos + len(replaced_text)
                    
                    if replaced_text.strip():
                        annotation_rows.append([
                            doc_name,   # File
                            i,          # Line (row index)
                            begin_offset,
                            end_offset,
                            entity_type
                        ])
                    
                    filled_pos = end_offset
                
                else:
                    # Instead of simply using find(), check if the filled text ends with the next chunk.
                    # This avoids capturing an occurrence of next_chunk that is embedded within the replaced text.
                    if filled_text.endswith(next_chunk):
                        next_chunk_pos = filled_text.rfind(next_chunk, filled_pos)
                    else:
                        next_chunk_pos = filled_text.find(next_chunk, filled_pos)
                    
                    if next_chunk_pos == -1:
                        # If not found, assume the replaced text is everything to the end
                        replaced_text = filled_text[filled_pos:]
                        begin_offset  = filled_pos
                        end_offset    = filled_pos + len(replaced_text)
                        
                        if replaced_text.strip():
                            annotation_rows.append([
                                doc_name,
                                i,
                                begin_offset,
                                end_offset,
                                entity_type
                            ])
                        filled_pos = end_offset
                    else:
                        # The replaced text is everything from filled_pos up to where next_chunk starts
                        replaced_text = filled_text[filled_pos:next_chunk_pos]
                        begin_offset  = filled_pos
                        end_offset    = next_chunk_pos
                        
                        if replaced_text.strip():
                            annotation_rows.append([
                                doc_name,
                                i,
                                begin_offset,
                                end_offset,
                                entity_type
                            ])
                        
                        filled_pos = next_chunk_pos
            
            else:
                # This is a normal text chunk (not a placeholder).
                pos = filled_text.find(part, filled_pos)
                if pos != -1:
                    filled_pos = pos + len(part)
                else:
                    pass

    ann_df = pd.DataFrame(annotation_rows, columns=["File", "Line", "Begin Offset", "End Offset", "Type"])
    
    return ann_df

# Transform the DataFrame and save the resulting CSV
annotations_df = transform_dataframe(aws_prepping)
print(annotations_df.head())
print("Unique types:", annotations_df["Type"].unique())
print(len(annotations_df["Type"].unique()))
print(len(annotations_df))
print(annotations_df["Type"].value_counts())


Row 25 details:
{'Template': 'Please create a legal analysis of the sports betting laws in [LOCATION] and send it to [EMAIL].', 'Filled Template': 'Please create a legal analysis of the sports betting laws in Montana and send it to Nichole.Kulas96@gmail.com.'}
Parts: ['Please create a legal analysis of the sports betting laws in ', '[LOCATION]', ' and send it to ', '[EMAIL]', '.']
               File  Line  Begin Offset  End Offset  Type
0  aws_ann_data.csv     0            91          94  NAME
1  aws_ann_data.csv     0            95         109  NAME
2  aws_ann_data.csv     0           114         120  NAME
3  aws_ann_data.csv     0           121         130  NAME
4  aws_ann_data.csv     1            29          61  NAME
Unique types: ['NAME' 'LOCATION' 'URL' 'USERNAME' 'JOB' 'EMAIL' 'GENDER' 'IP' 'PASSWORD'
 'CREDITCARDNUMBER' 'MASKEDNUMBER' 'USERAGENT' 'IBAN' 'CURRENCY'
 'CURRENCYNAME' 'BIC']
16
83921
Type
NAME                55856
LOCATION            12118
EMAIL                7914

In [ ]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(df_full["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
18,NAME,55856
3,LOCATION,12118
2,EMAIL,7914
23,JOB,3369
1,MASKEDNUMBER,1477
6,URL,1028
17,USERNAME,713
11,IP,557
10,PASSWORD,289
19,GENDER,255


In [ ]:
def mask_text_using_annotations(text, anns):
    # Sort annotations in reverse order of Begin Offset to avoid shifting issues.
    for ann in sorted(anns, key=lambda x: x["Begin Offset"], reverse=True):
        start = ann["Begin Offset"]
        end = ann["End Offset"]
        
        # Check for valid offset ranges.
        if start < 0 or end > len(text):
            print(f"Warning: Annotation offsets {start}-{end} are out of bounds for text length {len(text)}")
            continue

        # Replace the span with the masked placeholder.
        text = text[:start] + f"[{ann['Type']}]" + text[end:]
    
    return text

def apply_masking(row):
    # Retrieve all annotations corresponding to the current row.
    anns = annotations_df[annotations_df["Line"] == row.name].to_dict(orient="records")
    if anns:
        return mask_text_using_annotations(row["Filled Template"], anns)
    return row["Filled Template"]

tqdm.pandas(desc="Applying masking")
aws_prepping["Masked Filled Template"] = df_full.progress_apply(apply_masking, axis=1)


Applying masking:   0%|          | 24/42759 [00:00<03:04, 231.66it/s]

Applying masking: 100%|██████████| 42759/42759 [01:41<00:00, 420.98it/s]


In [ ]:
annotations_df[annotations_df["Line"] == id]

,File,Line,Begin Offset,End Offset,Type


In [ ]:
annotations_df

,File,Line,Begin Offset,End Offset,Type
0,aws_ann_data.csv,0,91,94,NAME
1,aws_ann_data.csv,0,95,109,NAME
2,aws_ann_data.csv,0,114,120,NAME
3,aws_ann_data.csv,0,121,130,NAME
4,aws_ann_data.csv,1,29,61,NAME
...,...,...,...,...,...
83934,aws_ann_data.csv,42756,46,61,EMAIL
83935,aws_ann_data.csv,42757,51,64,NAME
83936,aws_ann_data.csv,42758,58,65,NAME
83937,aws_ann_data.csv,42758,66,72,NAME


In [ ]:
id = 3218

In [ ]:
aws_prepping["Masked Filled Template"][id]

'Could you help me design a landing page for [NAME]'

In [ ]:
aws_prepping["Template"][id]

"Could you help me design a landing page for [NAME]'s upcoming digital marketing campaign? The website URL is [URL]."

In [ ]:
aws_prepping["Filled Template"][id]

"Could you help me design a landing page for Dickens - Fay's upcoming digital marketing campaign? The websitehttps://imperfect-railroad.name/ [URL_1]."

In [ ]:
aws_prepping.head()

,Template,Filled Template,Masked Filled Template
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","In our video conference, discuss the role of e..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis...",Could you draft a letter for [NAME] to send to...
2,Discuss the options for [NAME] [NAME] who want...,Discuss the options for Jeffery Pfeffer who wa...,Discuss the options for [NAME] [NAME] who want...
3,13. Write a press release announcing [NAME] [N...,13. Write a press release announcing Gayle Wat...,13. Write a press release announcing [NAME] [N...
4,9. Develop an inventory management plan for [N...,9. Develop an inventory management plan for Ev...,9. Develop an inventory management plan for [N...


In [ ]:
list_of_weird_rows = [42606,39778,39591,38894,35859,35398,34377,33200,32786,32633,31525,31171,30957,30419,30129,28097,25492,25335,24531,24497,23920,22670,22152,19842,19644,19461,18718,16604,16297,16239,15847,15775,15549,14604,13642,13103,12358,11358,10344,8652,738,6801,5071,4265,3654,3381,3218,1906,38778,36440,35086,7368,13681,29797, 7384, 13709, 29863, 35170, 36484, 38822,14604,30957, 36484]

In [ ]:
len(list_of_weird_rows)

# remove rows with the index from list of wierd rows
aws_prepping = aws_prepping.drop(list_of_weird_rows, errors='ignore')
aws_prepping.reset_index(drop=True)
aws_prepping

,Template,Filled Template,Masked Filled Template
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","In our video conference, discuss the role of e..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis...",Could you draft a letter for [NAME] to send to...
2,Discuss the options for [NAME] [NAME] who want...,Discuss the options for Jeffery Pfeffer who wa...,Discuss the options for [NAME] [NAME] who want...
3,13. Write a press release announcing [NAME] [N...,13. Write a press release announcing Gayle Wat...,13. Write a press release announcing [NAME] [N...
4,9. Develop an inventory management plan for [N...,9. Develop an inventory management plan for Ev...,9. Develop an inventory management plan for [N...
...,...,...,...
42754,Write a blog post for [NAME] about the role of...,Write a blog post for Stanton LLC about the ro...,Write a blog post for [NAME] about the role of...
42755,14. Calculate the return on investment for [NA...,14. Calculate the return on investment for Con...,14. Calculate the return on investment for [NA...
42756,Please write an email to [NAME] [NAME] at [EMA...,Please write an email to Roberta Gutmann V at ...,Please write an email to [NAME] [NAME] at [EMA...
42757,Can you help me write a project closure report...,Can you help me write a project closure report...,Can you help me write a project closure report...


In [ ]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(aws_prepping["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
18,NAME,55826
3,LOCATION,12114
2,EMAIL,7913
23,JOB,3369
1,MASKEDNUMBER,1467
6,URL,996
17,USERNAME,713
11,IP,541
10,PASSWORD,289
19,GENDER,255


In [ ]:
# Remove the list of weird rows from aws_prepping
# Remove rows from annotations_df where the "Line" column matches any value in list_of_weird_rows
annotations_df = annotations_df[~annotations_df["Line"].isin(list_of_weird_rows)]
# Reset the index of annotations_df after removing rows
annotations_df = annotations_df.reset_index(drop=True)
annotations_df

,File,Line,Begin Offset,End Offset,Type
0,aws_ann_data.csv,0,91,94,NAME
1,aws_ann_data.csv,0,95,109,NAME
2,aws_ann_data.csv,0,114,120,NAME
3,aws_ann_data.csv,0,121,130,NAME
4,aws_ann_data.csv,1,29,61,NAME
...,...,...,...,...,...
83829,aws_ann_data.csv,42756,46,61,EMAIL
83830,aws_ann_data.csv,42757,51,64,NAME
83831,aws_ann_data.csv,42758,58,65,NAME
83832,aws_ann_data.csv,42758,66,72,NAME


In [ ]:
# Compare the two strings
counter = 0
for i, row in aws_prepping.iterrows():
    masked = aws_prepping["Masked Filled Template"][i]
    template = aws_prepping["Template"][i]

    # Check if they are equal
    if masked != template:
        print("="*80)
        print(f"Masked Filled Template at index {i}:")
        print(masked)
        print(f"\nTemplate at index {i}:")
        print(template)
        print(f"\nOrginal Filled Template at index {i}:")
        print(aws_prepping["Filled Template"][i])
        print("\nThey are different.")
        counter += 1

print(f"errors in masking: {counter}")
        

errors in masking: 0


In [ ]:
aws_prepping

,Template,Filled Template,Masked Filled Template
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","In our video conference, discuss the role of e..."
1,Could you draft a letter for [NAME] to send to...,"Could you draft a letter for Dietrich, Schulis...",Could you draft a letter for [NAME] to send to...
2,Discuss the options for [NAME] [NAME] who want...,Discuss the options for Jeffery Pfeffer who wa...,Discuss the options for [NAME] [NAME] who want...
3,13. Write a press release announcing [NAME] [N...,13. Write a press release announcing Gayle Wat...,13. Write a press release announcing [NAME] [N...
4,9. Develop an inventory management plan for [N...,9. Develop an inventory management plan for Ev...,9. Develop an inventory management plan for [N...
...,...,...,...
42754,Write a blog post for [NAME] about the role of...,Write a blog post for Stanton LLC about the ro...,Write a blog post for [NAME] about the role of...
42755,14. Calculate the return on investment for [NA...,14. Calculate the return on investment for Con...,14. Calculate the return on investment for [NA...
42756,Please write an email to [NAME] [NAME] at [EMA...,Please write an email to Roberta Gutmann V at ...,Please write an email to [NAME] [NAME] at [EMA...
42757,Can you help me write a project closure report...,Can you help me write a project closure report...,Can you help me write a project closure report...


#### data loss is minimal

In [117]:
aws_prepping.to_csv("aws_prepping.csv", index=False)
annotations_df.to_csv("aws_ann_data.csv", index=False)


## AWS Comprehend dataprepping END

In [ ]:
from IPython.display import clear_output

for entities in unique_matches:
	regex_pattern = r'\[' + entities + r'\]'
	sextype_examples = df_full[df_full['Template'].str.contains(regex_pattern, na=False)]

	if sextype_examples.empty:
		print("No rows found with the token [SEXTYPE]")
	else:
		num_rows = min(20, len(sextype_examples))
		for i in range(num_rows):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)
	input("Press Enter to load next value...")
	clear_output(wait=True)
	

# Short form removal

In [ ]:
# contraction expansion

def remove_short_form(text):
    if isinstance(text, str):
        contractions = {
            # Common negatives
            "isn't": "is not",
            "wasn't": "was not",
            "weren't": "were not",
            "haven't": "have not",
            "hasn't": "has not",
            "hadn't": "had not",
            "don't": "do not",
            "doesn't": "does not",
            "didn't": "did not",
            "won't": "will not",
            "wouldn't": "would not",
            "shouldn't": "should not",
            "couldn't": "could not",
            "mustn't": "must not",
            "shan't": "shall not",
            
            # Pronoun contractions
            "i'm": "i am",
            "you're": "you are",
            "he's": "he is",
            "she's": "she is",
            "it's": "it is",
            "we're": "we are",
            "they're": "they are",

            # Possession / descriptive
            "who's": "who is",
            "what's": "what is",
            "where's": "where is",
            "when's": "when is",
            "why's": "why is",
            "how's": "how is",
            "there's": "there is",
            "here's": "here is",
            "that's": "that is",

            # Past and hypothetical
            "i'd": "i would",
            "you'd": "you would",
            "he'd": "he would",
            "she'd": "she would",
            "we'd": "we would",
            "they'd": "they would",

            # Future forms
            "i'll": "i will",
            "you'll": "you will",
            "he'll": "he will",
            "she'll": "she will",
            "it'll": "it will",
            "we'll": "we will",
            "they'll": "they will",

            # Perfect tense
            "i've": "i have",
            "you've": "you have",
            "we've": "we have",
            "they've": "they have",
            "he's": "he has",
            "she's": "she has",
            "it's": "it has",

            # Imperative/other
            "let's": "let us",
            "y'all": "you all",
            "o'clock": "of the clock",
            "ma'am": "madam",
            "gonna": "going to",
            "wanna": "want to",
            "gotta": "got to",
            "ain't": "is not"
        }

        # Replace contractions case-insensitively
        for contraction, replacement in contractions.items():
            text = re.sub(rf"\b{re.escape(contraction)}\b", replacement, text, flags=re.IGNORECASE)

        return text
    return text



# Normalization

In [ ]:
def normalize_text(text):
	# text to lower
	parts = re.split(r'(\[[^\]]*\])', text)

	processed_parts = []
	for segment in parts:
		if segment.startswith('[') and segment.endswith(']'):
			processed_parts.append(segment)
		else:
			processed_parts.append(segment.lower())

	text = "".join(processed_parts)
	# remove extra spaces
	text = re.sub(r'\s+', ' ', text)
	# Use the function to remove short forms
	text = remove_short_form(text)
	return text

In [ ]:
from tqdm import tqdm

tqdm.pandas(desc="Normalizing Template")
df_full['Template'] = df_full['Template'].progress_apply(normalize_text)
tqdm.pandas(desc="Normalizing Filled Template")
df_full['Filled Template'] = df_full['Filled Template'].progress_apply(normalize_text)

# Lemming

In [ ]:

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")# TODO: Make this use the large model
spacy.prefer_gpu()

def lemmatize_text(text):
    # Check for missing values
    if pd.isna(text):
        return text
    # Process the text with spaCy
    doc = nlp(text)
    # Join lemmatized tokens back into a string
    return " ".join(token.lemma_ for token in doc)

In [ ]:

try:
	# Try loading the lemmatized data
	df_full = pd.read_csv("lemmatized_data.csv")
	print("Loaded data from lemmatized_data.csv")
except FileNotFoundError:
	# If the file doesn't exist, perform lemmatization
	print("lemmatized_data.csv not found, performing lemmatization...")
	tqdm.pandas(desc="Lemmatizing Template")
	df_full['Template'] = df_full['Template'].progress_apply(lemmatize_text)
	tqdm.pandas(desc="Lemmatizing Filled Template")
	df_full['Filled Template'] = df_full['Filled Template'].progress_apply(lemmatize_text)

	# Check the results
	print(df_full.head())
	#save
	df_full.to_csv("lemmatized_data.csv", index=False)

In [ ]:
# Count the labeled items in the DataFrame
total_labeled_items = count_labeled_items(df_full)
print(f"Total number of labeled items: {total_labeled_items}")

# Sentences Splitting

In [ ]:
def split_into_sentences(text):
    """
    Splits a given text (string) into a list of sentences using spaCy's sentence segmentation.
    Returns an empty list if the input is None or NaN.
    """
    if pd.isna(text):
        return []
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [ ]:
import os

if not os.path.exists("sentences_data.csv"):
	tqdm.pandas(desc="Splitting Sentences")
	
	df_full["Template_Sentences"] = df_full["Template"].progress_apply(split_into_sentences)
	df_full["Filled_Sentences"]   = df_full["Filled Template"].progress_apply(split_into_sentences)
	
	# Optionally, save the updated dataframe
	df_full.to_csv("sentence_data.csv", index=False)
	
	print("Sentence splitting complete. Data saved to sentence_data.csv")
	print(df_full.head())
else:
	print("sentence_data.csv already exists. Skipping sentence splitting.")


## VIEW DATA

In [ ]:
df_full.head()

In [ ]:
from IPython.display import clear_output

for entities in unique_matches:
	print(entities)

	entities = entities.lower()
	print(entities)
	regex_pattern = r'\[ ' + entities + r' \]'
	sextype_examples = df_full[df_full['Template'].str.contains(regex_pattern, na=False)]

	if sextype_examples.empty:
		print("No rows found with the token")
	else:
		num_rows = min(20, len(sextype_examples))
		for i in range(num_rows):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)
	input("Press Enter to load next value...")
	clear_output(wait=True)
	

In [ ]:

# Create 10 datasets with test percentages from 10% to 100%
test_pcts = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]

# Build list of datasets
df_list = [get_test_set(df_full, pct, random_state=42) for pct in test_pcts]


datasets = {}
for name, ds in zip(test_pcts, df_list):
    # Create a dataset instance with default (empty) init values.
    result = dataset(name=name, dataset=ds)
    datasets[name] = result
datasets[0.1].dataset.head()

### NLP Implementation 

In [ ]:
analyzer = AnalyzerEngine()

In [ ]:
# test Call analyzer to get results
results = analyzer.analyze(text="My phone number is 212-555-5555",
                           entities=["PHONE_NUMBER"],
                           language='en')
print(results)

In [ ]:
def clean_text(text):
    """Clean and normalize text."""
    return str(text).strip()

def extract_entities(template, filled):
    """
    Extract entities by aligning a template (with placeholders) to the filled text.
    
    Assumes that the filled text is identical to the template except that each placeholder 
    (e.g. "[NAME_1]") has been replaced by the actual value.
    
    Returns:
        A list of tuples (start_char, end_char, label) for entities in the filled text.
    """
    entities = []
    i = 0  # pointer for template
    j = 0  # pointer for filled text

    while i < len(template) and j < len(filled):
        if template[i] == '[':
            # Found a placeholder in the template.
            closing = template.find(']', i)
            if closing == -1:
                break  # malformed template (no matching ])
            # Extract the raw placeholder, e.g. "[NAME_1]"
            # Remove the brackets and any trailing digits to get the label.
            label_raw = template[i+1:closing]    # e.g. "NAME_1"
            label = re.sub(r'_\d+', '', label_raw)  # e.g. becomes "NAME"
            
            # Determine the literal text that follows the placeholder in the template.
            next_i = closing + 1
            next_bracket = template.find('[', next_i)
            literal = template[next_i:] if next_bracket == -1 else template[next_i:next_bracket]
            
            # In the filled text, the actual entity value replaces the placeholder.
            # We assume that the literal following the placeholder appears unchanged.
            if literal:
                literal_index = filled.find(literal, j)
            else:
                literal_index = len(filled)
            
            if literal_index == -1:
                # If we cannot find the literal, assume the entity is the rest of the filled text.
                entity_start = j
                entity_end = len(filled)
                j = len(filled)
            else:
                entity_start = j
                entity_end = literal_index
                j = literal_index  # advance pointer j to the beginning of the literal
            
            entities.append((entity_start, entity_end, label))
            # Advance pointer i past the entire placeholder.
            i = closing + 1
        else:
            # For non-placeholder characters, assume they match between template and filled.
            if template[i] == filled[j]:
                i += 1
                j += 1
            else:
                # If there is a mismatch (e.g. extra whitespace), increment j.
                j += 1
    return entities


In [ ]:
# TODO: Run later: i want to collect F1 score, accuracy, false postetives, false negatives etc. everything

def NLP_training(df):
    # Build initial training data by aligning each template with its filled version.
    raw_train_data = []
    for _, row in df.iterrows():
        template = clean_text(row['Template'])
        filled = clean_text(row['Filled Template'])
        entities = extract_entities(template, filled)
        if entities:
            # Each training example is a tuple: (text, {"entities": [(start, end, label), ...]})
            raw_train_data.append((filled, {"entities": entities}))

    # ------------------------------
    # Step 0.5: Re-align Entity Offsets to Token Boundaries
    # ------------------------------
    tokenizer_nlp = spacy.blank("en")
    aligned_train_data = []
    for text, annotation in raw_train_data:
        doc = tokenizer_nlp(text)
        new_entities = []
        for start, end, label in annotation["entities"]:
            # Use "expand" mode to adjust the span to token boundaries.
            span = doc.char_span(start, end, alignment_mode="expand")
            if span is not None:
                new_entities.append((span.start_char, span.end_char, label))
            else:
                # If alignment fails, you might choose to log or skip the entity.
                print(f"WARNING: Could not align entity '{text[start:end]}' in text: {text}")
        if new_entities:
            aligned_train_data.append((text, {"entities": new_entities}))
    # Use the aligned data for training.
    TRAIN_DATA = aligned_train_data

    # ------------------------------
    # Step 1: Split Data
    # ------------------------------
    # Here we use an 80/20 train/validation split.
    train_size = int(0.8 * len(TRAIN_DATA))
    train_data = TRAIN_DATA[:train_size]
    valid_data = TRAIN_DATA[train_size:]

    # ------------------------------
    # Step 2: Create and Configure the Model
    # ------------------------------
    nlp = spacy.blank("en")

    # Add a Named Entity Recognizer (NER) pipeline component if not already present.
    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner", last=True)
    else:
        ner = nlp.get_pipe("ner")

    # Add each entity label from the training data to the NER component.
    for _, annotations in train_data:
        for start, end, label in annotations["entities"]:
            ner.add_label(label)

    # ------------------------------
    # Step 3: Train the Model Using Batches with Dropout
    # ------------------------------
    optimizer = nlp.begin_training()
    n_iter = 20  # Number of epochs
    batch_size = 16

    for itn in range(n_iter):
        random.shuffle(train_data)
        batches = minibatch(train_data, size=batch_size)
        losses = {}
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                examples.append(Example.from_dict(doc, annotations))
            nlp.update(examples, sgd=optimizer, drop=0.3, losses=losses)
        print(f"Iteration {itn + 1}/{n_iter} - Losses: {losses}")

    # ------------------------------
    # Step 4: Define Improved Masking Function
    # ------------------------------
    def mask_pii(text, model):
        """
        Mask detected entities in the text with their label names.
        
        Entities are replaced starting from the end of the text (to avoid offset issues).
        """
        doc = model(text)
        spans = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
        # Sort spans in reverse order of start index.
        spans = sorted(spans, key=lambda x: x[0], reverse=True)
        masked_text = text
        for start, end, label in spans:
            masked_text = masked_text[:start] + label + masked_text[end:]
        return masked_text

    # ------------------------------
    # Step 5: Evaluate and Print Combined Results
    # ------------------------------
    correct_texts = 0
    correct_labels = 0
    total_texts = 0
    total_labels = 0
    failed_labels = []

    print("\n=== Evaluation on Validation Data ===\n")
    for text, annotations in valid_data:
        masked_text = mask_pii(text, nlp)
        
        print("Original Text:")
        print(text)
        print("Masked Text:")
        print(masked_text)
        print("-" * 40)
        
        text_correct = True
        # Evaluate masking on each individual entity.
        for start, end, label in annotations["entities"]:
            total_labels += 1
            if label in masked_text:
                correct_labels += 1
            else:
                text_correct = False
                failed_labels.append((label, text[start:end]))
        
        if text_correct:
            correct_texts += 1
        total_texts += 1

    if failed_labels:
        print("\nFAILED MASKINGS:")
        for label, value in failed_labels:
            print(f"Label: {label}, Expected Value: {value}")
    else:
        print("\nAll entities were successfully masked in every text!")

    text_accuracy = correct_texts / total_texts if total_texts > 0 else 0
    label_accuracy = correct_labels / total_labels if total_labels > 0 else 0

    print(f"\nText Accuracy (all entities in a text masked correctly): {text_accuracy:.2%}")
    print(f"Label Accuracy (individual entity masking): {label_accuracy:.2%}")
    return text_accuracy, label_accuracy, failed_labels, nlp


In [ ]:

for key, ds_obj in datasets.items():
    print(f"Processing dataset with test percentage: {ds_obj.name}")
    text_acc, label_acc, failed_labels, nlp_model = NLP_training(ds_obj.dataset)
    ds_obj.text_accuracy = text_acc
    ds_obj.label_accuracy = label_acc
    ds_obj.nlp = nlp_model
    print(ds_obj)

    with open(f"./tests/dataset_{ds_obj.name}.pkl", "wb") as file:
        pickle.dump(ds_obj, file)
    print(f"Saved ds_obj to dataset_{ds_obj.name}.pkl")

In [ ]:

# Extract sorted list of dataset keys (test percentages)
sorted_keys = sorted(datasets.keys())

# Get accuracy values (convert to percentage)
text_accs = [datasets[k].text_accuracy * 100 for k in sorted_keys]
label_accs = [datasets[k].label_accuracy * 100 for k in sorted_keys]

# Create the plot
plt.figure(figsize=(8, 5))
plt.plot(sorted_keys, text_accs, marker='o', label='Text Accuracy')
plt.plot(sorted_keys, label_accs, marker='o', label='Label Accuracy')
plt.xlabel('Test Percentage')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Comparison Across Datasets')
plt.legend()
plt.grid(True)
plt.show()